# 第四次作业：sklearn 线性回归源码解析（主线：OLS + SGD）

> 主题：阅读 scikit-learn 的实现，看课堂上没讲透的「工程决策」。
> 覆盖范围仅**主线**：OLS（`LinearRegression`）与 SGD 梯度下降（`SGDRegressor`）；Ridge / Lasso / GLM / Huber 不在本篇。

环境：`scikit-learn 1.3.2` / Python 3.8。文中行号均以 **1.3.2** 版本的库为准；code cell 一律用 `inspect.getsource` **现场取源码**，不写死行号。


## 1. 线性回归的两张「面孔」：课上 vs sklearn

课堂上的线性回归面貌很简洁：给定 `X`、`y`，最小二乘的正规方程是

$$w = (X^\top X)^{-1} X^\top y$$

（带截距时用增广矩阵或先去均值），正则化版本再加一项 `α‖w‖²`（岭）或 `α‖w‖₁`（Lasso）。这些公式本体本作业**不展开**。

两张面孔的差别在于：课上讲的是**数学上正确的解**；sklearn 实现里串起的是一连串**工程决策**——同一份数学，为了适应稀疏数据、多目标、正约束、大样本与在线学习，被拆成了多条路径、多种表示。

本作业只挑 sklearn 实现里「不一样」的地方看，按两条主线组织：

| 主线 | 源码 | 关键工程决策 |
|---|---|---|
| **OLS**（`LinearRegression`） | `linear_model/_base.py` | ① **三类后端分派**（NNLS / 稀疏 LSQR / 稠密 SVD）；② **截距去均值重构**；③ 稀疏数据的**隐式中心化算子**；④ 样本权重的 **√w 重缩放** |
| **SGD**（`SGDRegressor`） | `linear_model/_stochastic_gradient.py` | ⑤ **Python 配置台 + C 内循环**（字符串 → 枚举 → 编译模块）；⑥ 优化状态（`t_` / ASGD / 验证掩码 / 截距衰减） |

一句话贯穿全篇：**分派条件（`dtype` / 形状 / 稀疏 / `positive` / 多目标）决定算法，数值稳定性由 scipy 后端兜底。**


## 2. 定位源码：`linear_model` 的可分析边界

与作业 2（KNN 大量 `.so`）、作业 3（朴素贝叶斯几乎纯 Python）不同，本次是**混合结构**：

- **纯 Python，可 `inspect.getsource` 逐行读**：`_base.py`（OLS）、`_stochastic_gradient.py`（SGD 的**外层流程**）、`_glm/`、`_linear_loss.py` 等；
- **本地包内只有 `.so`，看不到内循环**：`_cd_fast`（坐标下降）、`_sag_fast`（SAG/SAGA）、`_sgd_fast`（SGD 内循环 `_plain_sgd32/64` 与各 `LossFunction` / `Penalty` / `LearningRate` 类）。

对编译模块只能做「接口 / 分派点 / docstring / 调用点」的**黑盒分析**；如需深入实现，只能另读对应版本的上游 Cython `.pyx`——那不属于本项目可 dump 的范围。

本作业要讲清的就是这条边界：**Python 侧是「配置台与分派台」，真正的数值循环在 C 层。**


In [1]:
import os
import sys

import sklearn

sk_dir = os.path.dirname(sklearn.__file__)
lm_dir = os.path.join(sk_dir, "linear_model")

print("python       :", sys.version.split()[0])
print("sklearn      :", sklearn.__version__)
print("sklearn path :", sk_dir)
print("linear_model :", lm_dir)
print()

pure, compiled = [], []
for f in sorted(os.listdir(lm_dir)):
    if f.endswith(".py"):
        pure.append(f)
    elif f.endswith(".so"):
        compiled.append(f)

print("== 纯 Python（可 inspect.getsource 逐行读）==")
for f in pure:
    print("   ", f)

print()
print("== 编译产物 .so（只有二进制，内循环看不到源码）==")
for f in compiled:
    print("   ", f)


python       : 3.8.20
sklearn      : 1.3.2
sklearn path : /root/MSA/ENTER/envs/vmunet/lib/python3.8/site-packages/sklearn
linear_model : /root/MSA/ENTER/envs/vmunet/lib/python3.8/site-packages/sklearn/linear_model

== 纯 Python（可 inspect.getsource 逐行读）==
    __init__.py
    _base.py
    _bayes.py
    _coordinate_descent.py
    _huber.py
    _least_angle.py
    _linear_loss.py
    _logistic.py
    _omp.py
    _passive_aggressive.py
    _perceptron.py
    _quantile.py
    _ransac.py
    _ridge.py
    _sag.py
    _stochastic_gradient.py
    _theil_sen.py

== 编译产物 .so（只有二进制，内循环看不到源码）==
    _cd_fast.cpython-38-x86_64-linux-gnu.so
    _sag_fast.cpython-38-x86_64-linux-gnu.so
    _sgd_fast.cpython-38-x86_64-linux-gnu.so


## 3. OLS 不用正规方程：三类后端 × 多目标策略（R1/R5/R6/R7）

`LinearRegression` 的 docstring 自己就点破了（`_base.py` L608–610）：

> *"From the implementation point of view, this is just plain Ordinary Least Squares (**scipy.linalg.lstsq**) or Non Negative Least Squares (**scipy.optimize.nnls**) wrapped as a predictor object."*

也就说：教科书里的正规方程 `w=(XᵀX)⁻¹Xᵀy` **从不被使用**。`fit`（L709–752）实际是**三类后端分派**：

| # | 触发条件 | 后端 | 备注 |
|---|---|---|---|
| ① | `positive=True` | `scipy.optimize.nnls`（active-set） | 非负最小二乘，另一套算法 |
| ② | `X` 稀疏（`csr` / `csc` / `coo`） | `scipy.sparse.linalg.lsqr`（LSQR 迭代） | 配合第 4 节的隐式中心化 |
| ③ | 稠密且 `positive=False` | `scipy.linalg.lstsq`（**SVD**） | 数值最稳，还顺带给出秩与奇异值 |

**多目标不是第 4 个后端，而是一种「策略」**：稠密 `lstsq` **原生支持 2D `y`**（返回 `(n_features, n_targets)` 再转置）；只有 `nnls` 与稀疏 `lsqr` 因为**只吃 1D `y`**，才**逐列** `joblib.Parallel` 拼回结果。

**`rank_` / `singular_` 只在「稠密 且 `positive=False` 且走 `lstsq` 分支」时赋值**——它们来自 `linalg.lstsq` 的返回值；稀疏 LSQR 与 NNLS 路径拿不到，且 `positive=True` 甚至在 `_validate_data` 阶段就禁掉了稀疏（L676）。属性文档只说「仅当 `X` 稠密时可用」，上面这句是它的**精确化**。

**`n_jobs` 的触发条件：多目标 且（稀疏 或 `positive=True`）**——它是 docstring（L552–558）里一个「条件性参数」，实现中**唯一**的使用处就是上面那两处 `Parallel`。条件不满足时不走 joblib，但这**不代表底层只有一个线程**：BLAS / LAPACK 仍可能多线程。


In [2]:
import inspect

import numpy as np
from scipy.sparse import csr_matrix
from sklearn.linear_model import LinearRegression

Q = '"' * 3


def strip_docstring(src):
    # 删掉函数最外层 docstring，控制篇幅
    out, in_doc = [], False
    for line in src.splitlines():
        s = line.strip()
        if not in_doc:
            if s.startswith(Q):
                in_doc = s.count(Q) < 2
                continue
            out.append(line)
        else:
            if s.endswith(Q):
                in_doc = False
            continue
    return out


# 关键位置的注释（不写行号，只标「位置」），与 markdown 中的结论一一对应
MARKS = {
    "if self.positive:":
        "  # <== 后端①：positive=True -> scipy.optimize.nnls",
    "elif sp.issparse(X):":
        "  # <== 后端②：稀疏 X -> scipy.sparse.linalg.lsqr（LSQR）",
    "self.coef_, _, self.rank_, self.singular_ = linalg.lstsq(X, y)":
        "  # <== 后端③：稠密 -> SVD；rank_/singular_ 只在这里产生",
    "outs = Parallel(n_jobs=n_jobs_)(":
        "  # <== 多目标「策略」：只吃 1D y 的后端才逐列并行",
    "X_centered = sparse.linalg.LinearOperator(":
        "  # <== 隐式中心化算子（不物化中心化矩阵）",
    "self._set_intercept(X_offset, y_offset, X_scale)":
        "  # <== 由偏移量反推 intercept_",
}

print("# LinearRegression.fit()（已删 docstring）—— 三类后端分派 + 多目标策略")
for line in strip_docstring(inspect.getsource(LinearRegression.fit)):
    print(line + MARKS.get(line.strip(), ""))


# LinearRegression.fit()（已删 docstring）—— 三类后端分派 + 多目标策略
    @_fit_context(prefer_skip_nested_validation=True)
    def fit(self, X, y, sample_weight=None):
        n_jobs_ = self.n_jobs

        accept_sparse = False if self.positive else ["csr", "csc", "coo"]

        X, y = self._validate_data(
            X, y, accept_sparse=accept_sparse, y_numeric=True, multi_output=True
        )

        has_sw = sample_weight is not None
        if has_sw:
            sample_weight = _check_sample_weight(
                sample_weight, X, dtype=X.dtype, only_non_negative=True
            )

        # Note that neither _rescale_data nor the rest of the fit method of
        # LinearRegression can benefit from in-place operations when X is a
        # sparse matrix. Therefore, let's not copy X when it is sparse.
        copy_X_in_preprocess_data = self.copy_X and not sp.issparse(X)

        X, y, X_offset, y_offset, X_scale = _preprocess_data(
            X,
            y,
            fit_intercep

In [3]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.linear_model import LinearRegression

rng = np.random.RandomState(0)
X = rng.randn(200, 5)                     # 条件良好：n_samples=200 >> n_features=5
y = X @ np.array([1.5, -2.0, 0.5, 3.0, -1.0]) + 0.7 + 0.01 * rng.randn(200)

# --- 稠密 vs 稀疏：同一份数据，两条后端 ---
m_dense = LinearRegression().fit(X, y)
m_sparse = LinearRegression().fit(csr_matrix(X), y)
print("稠密 lstsq  coef_ :", np.round(m_dense.coef_, 6))
print("稀疏 lsqr   coef_ :", np.round(m_sparse.coef_, 6))
print("coef_ 一致(allclose):", np.allclose(m_dense.coef_, m_sparse.coef_, atol=1e-6))
print("截距一致            :", np.allclose(m_dense.intercept_, m_sparse.intercept_, atol=1e-6))

# --- rank_ / singular_：只有「稠密 + positive=False + 走 lstsq」才有 ---
nnls_probe = LinearRegression(positive=True).fit(X, y)
print()
print("稠密模型的 rank_     :", m_dense.rank_)
print("稠密模型的 singular_ :", np.round(m_dense.singular_, 4))
print("稀疏模型 hasattr(rank_)     :", hasattr(m_sparse, "rank_"))
print("稀疏模型 hasattr(singular_) :", hasattr(m_sparse, "singular_"))
print("NNLS模型 hasattr(rank_)     :", hasattr(nnls_probe, "rank_"))

# --- positive=True 分两组：约束「不活跃」与「活跃」 ---
Xp = np.abs(rng.randn(200, 4)) * 0.5 + 0.01     # 构造让 OLS 解本就非负的数据

# ① 约束不活跃：OLS 解本来就非负 -> NNLS 与 OLS 完全一致
y_inactive = Xp @ np.array([1.0, 2.0, 0.5, 3.0]) + 0.3 + 0.01 * rng.randn(200)
ols1 = LinearRegression().fit(Xp, y_inactive)
nnls1 = LinearRegression(positive=True).fit(Xp, y_inactive)
print()
print("[① 约束不活跃] OLS  :", np.round(ols1.coef_, 6))
print("[① 约束不活跃] NNLS :", np.round(nnls1.coef_, 6))
print("    与 OLS 一致:", np.allclose(ols1.coef_, nnls1.coef_, atol=1e-6))

# ② 约束活跃：OLS 解含负系数 -> NNLS 把负系数截断，结果不同
y_active = Xp @ np.array([1.0, -2.0, 0.5, 3.0]) + 0.3 + 0.01 * rng.randn(200)
ols2 = LinearRegression().fit(Xp, y_active)
nnls2 = LinearRegression(positive=True).fit(Xp, y_active)
print()
print("[② 约束活跃] OLS  :", np.round(ols2.coef_, 6), "  <- 第 2 个系数为负，违反非负约束")
print("[② 约束活跃] NNLS :", np.round(nnls2.coef_, 6), "  <- 负系数被截断为 0")
print("    与 OLS 不同:", not np.allclose(ols2.coef_, nnls2.coef_, atol=1e-6),
      " 且 NNLS 解非负:", bool(np.all(nnls2.coef_ >= 0)))

# --- 多目标：稠密原生、稀疏逐列 ---
y2 = np.c_[y, 2 * y]
print()
print("多目标 y2.shape                  :", y2.shape)
print("稠密 coef_.shape :", LinearRegression().fit(X, y2).coef_.shape, " <- lstsq 原生支持 2D y")
print("稀疏 coef_.shape :", LinearRegression().fit(csr_matrix(X), y2).coef_.shape, " <- 逐列 lsqr 拼回")


稠密 lstsq  coef_ : [ 1.49961  -2.001446  0.499997  2.999666 -0.99987 ]
稀疏 lsqr   coef_ : [ 1.49961  -2.001446  0.499997  2.999666 -0.99987 ]
coef_ 一致(allclose): True
截距一致            : True

稠密模型的 rank_     : 5
稠密模型的 singular_ : [15.2621 14.4607 13.9477 13.5648 12.3145]
稀疏模型 hasattr(rank_)     : False
稀疏模型 hasattr(singular_) : False
NNLS模型 hasattr(rank_)     : False

[① 约束不活跃] OLS  : [0.997623 2.001927 0.502532 3.001673]
[① 约束不活跃] NNLS : [0.997623 2.001927 0.502532 3.001673]
    与 OLS 一致: True

[② 约束活跃] OLS  : [ 0.993113 -2.002491  0.499478  2.99867 ]   <- 第 2 个系数为负，违反非负约束
[② 约束活跃] NNLS : [1.046862 0.       0.4284   3.042586]   <- 负系数被截断为 0
    与 OLS 不同: True  且 NNLS 解非负: True

多目标 y2.shape                  : (200, 2)
稠密 coef_.shape : (2, 5)  <- lstsq 原生支持 2D y
稀疏 coef_.shape : (2, 5)  <- 逐列 lsqr 拼回


## 4. 截距：去均值重构 + 稀疏隐式中心化（R2/R3）

`fit_intercept=True` 时，截距**不是**「给 X 加一列 1」，而是（`_preprocess_data` L248–286）：

1. `X`、`y` 各自减去**加权均值**（有 `sample_weight` 时按权重求平均）；
2. 在**去均值后的数据**上拟合一个**无截距**模型；
3. 再由 `intercept_ = y_offset − X_offset · coef_` **反推**截距（`_set_intercept` L394）。

好处是**不必显式扩列**，SVD / LSQR 直接作用在原始列空间上。

一处容易误读的地方：本路径（`LinearRegression`）**未启用 `normalize`**，所以 `_preprocess_data` 里 `X_scale = np.ones(...)` —— **`X_scale` 恒为全 1**。因此 `_set_intercept` 里那句 `np.divide(coef_, X_scale)` 实际是**恒等操作**；它是为 `normalize=True` 的老路径与其它模型保留的统一写法。`fit_intercept=False` 时更直接：`intercept_ = 0.0`。

稀疏 `X` 还多一层：**真去中心化会破坏稀疏性**（减均值后几乎处处非零 → 变稠密）。于是 `fit`（L718–739）不物化中心化矩阵，而是用 `LinearOperator` 定义

- `matvec(b)  = X·b  − b·(X_offset / X_scale)`
- `rmatvec(b) = Xᵀ·b − (X_offset / X_scale)·b.sum()`

把「已中心化的 X」表示成一个**只提供矩阵向量乘的算子**喂给 `lsqr` —— **数学等价、内存不爆**。


In [4]:
import inspect

from sklearn.linear_model._base import LinearModel, _preprocess_data

# _preprocess_data 很长：只切「去均值 + X_scale」这一段
lines = inspect.getsource(_preprocess_data).splitlines()
s = next(k for k, l in enumerate(lines) if "if fit_intercept:" in l)
e = next(k for k, l in enumerate(lines) if "return X, y, X_offset, y_offset, X_scale" in l)

print("# _preprocess_data() —— 去均值与 X_scale（本路径下未启用 normalize）")
print("\n".join(lines[s:e + 1]))

print()
print("# LinearModel._set_intercept() —— 由偏移量反推 intercept_")
print(inspect.getsource(LinearModel._set_intercept))


# _preprocess_data() —— 去均值与 X_scale（本路径下未启用 normalize）
    if fit_intercept:
        if sp.issparse(X):
            X_offset, X_var = mean_variance_axis(X, axis=0, weights=sample_weight)
        else:
            if normalize:
                X_offset, X_var, _ = _incremental_mean_and_var(
                    X,
                    last_mean=0.0,
                    last_variance=0.0,
                    last_sample_count=0.0,
                    sample_weight=sample_weight,
                )
            else:
                X_offset = np.average(X, axis=0, weights=sample_weight)

            X_offset = X_offset.astype(X.dtype, copy=False)
            X -= X_offset

        if normalize:
            X_var = X_var.astype(X.dtype, copy=False)
            # Detect constant features on the computed variance, before taking
            # the np.sqrt. Otherwise constant features cannot be detected with
            # sample weights.
            constant_mask = _is_constant_feature(X_var, X_o

In [5]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.linear_model import LinearRegression
from sklearn.linear_model._base import _preprocess_data

rng = np.random.RandomState(0)
X = rng.randn(200, 5)
y = X @ np.array([1.5, -2.0, 0.5, 3.0, -1.0]) + 0.7 + 0.01 * rng.randn(200)

# 手工复现 fit 内部的去均值：拿 _preprocess_data 返回的偏移量
Xc, yc, X_offset, y_offset, X_scale = _preprocess_data(
    X.copy(), y.copy(), fit_intercept=True, copy=False, sample_weight=None
)

# 无截距模型在「去均值数据」上的解 == 有截距模型的 coef_
coef_centered = np.linalg.lstsq(Xc, yc, rcond=None)[0]
m = LinearRegression(fit_intercept=True).fit(X, y)

print("X_offset（各列加权均值）:", np.round(X_offset, 4))
print("y_offset               :", round(float(y_offset), 4))
print("去均值上无截距解的 coef_ == 模型 coef_ :", np.allclose(coef_centered, m.coef_, atol=1e-8))

# intercept_ = y_offset - X_offset · coef_
intercept_manual = y_offset - np.dot(X_offset, m.coef_)
print()
print("手工反推 intercept_ :", repr(float(intercept_manual)))
print("模型 intercept_     :", repr(float(m.intercept_)))
print("一致(allclose)      :", bool(np.allclose(intercept_manual, m.intercept_)))

# X_scale：本路径未启用 normalize -> 恒为全 1，_set_intercept 的除法是恒等操作
print()
print("X_scale            :", X_scale)
print("X_scale 是否全为 1 :", bool(np.all(X_scale == 1.0)))

# 稀疏 X 有截距：隐式中心化算子的预测与稠密一致
m_sparse = LinearRegression().fit(csr_matrix(X), y)
print()
print("稀疏 X 有截距模型 预测 == 稠密模型 预测 :",
      bool(np.allclose(m_sparse.predict(csr_matrix(X)), m.predict(X), atol=1e-6)))


X_offset（各列加权均值）: [-0.1005 -0.067  -0.0239 -0.0648  0.03  ]
y_offset               : 0.4477
去均值上无截距解的 coef_ == 模型 coef_ : True

手工反推 intercept_ : 0.700799132377522
模型 intercept_     : 0.700799132377522
一致(allclose)      : True

X_scale            : [1. 1. 1. 1. 1.]
X_scale 是否全为 1 : True

稀疏 X 有截距模型 预测 == 稠密模型 预测 : True


## 5. 样本权重：√w 重缩放把 WLS 折成 OLS（R4）

带权最小二乘（WLS）要最小化 `(y−Xw)ᵀ S (y−Xw)`，其中 `S = diag(sample_weight)`。sklearn 不为此单开一条求解路径，而是利用一个恒等式（`_rescale_data` docstring L308–312 直接写出）：

$$(y-Xw)^\top S\,(y-Xw) \;=\; \lVert \sqrt{S}\,y \;-\; \sqrt{S}\,X\,w \rVert_2^2$$

于是**加权最小二乘 ≡ 对 `√S·X, √S·y` 做普通最小二乘** —— 权重被吸收进数据本身。教科书只给 WLS 的正规方程，不会讲这层变换。

实现上还有个 `inplace` 优化（L338–341）：仅当 `_preprocess_data` 已经复制过数据时才原地缩放，避免多一次拷贝。


In [6]:
import inspect

from sklearn.linear_model._base import _rescale_data

print("# _rescale_data() —— 目标函数 (y-Xw)'S(y-Xw) == ||sqrt(S)y - sqrt(S)Xw||^2")
print(inspect.getsource(_rescale_data))


# _rescale_data() —— 目标函数 (y-Xw)'S(y-Xw) == ||sqrt(S)y - sqrt(S)Xw||^2
def _rescale_data(X, y, sample_weight, inplace=False):
    """Rescale data sample-wise by square root of sample_weight.

    For many linear models, this enables easy support for sample_weight because

        (y - X w)' S (y - X w)

    with S = diag(sample_weight) becomes

        ||y_rescaled - X_rescaled w||_2^2

    when setting

        y_rescaled = sqrt(S) y
        X_rescaled = sqrt(S) X

    Returns
    -------
    X_rescaled : {array-like, sparse matrix}

    y_rescaled : {array-like, sparse matrix}
    """
    # Assume that _validate_data and _check_sample_weight have been called by
    # the caller.
    n_samples = X.shape[0]
    sample_weight_sqrt = np.sqrt(sample_weight)

    if sp.issparse(X) or sp.issparse(y):
        sw_matrix = sparse.dia_matrix(
            (sample_weight_sqrt, 0), shape=(n_samples, n_samples)
        )

    if sp.issparse(X):
        X = safe_sparse_dot(sw_matrix, X)
    else:
  

In [7]:
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.RandomState(0)
X = rng.randn(60, 3)
y = X @ np.array([1.0, -3.0, 2.0]) + 1.0 + 0.05 * rng.randn(60)
sw = rng.randint(1, 4, size=60).astype(float)     # 整数权重，才方便「按权重重复」

m_weighted = LinearRegression().fit(X, y, sample_weight=sw)

# 等价变换：把每个样本按权重重复（重复 k 次 ≙ 权重 k）
idx = np.repeat(np.arange(60), sw.astype(int))
m_repeat = LinearRegression().fit(X[idx], y[idx])

print("sample_weight 拟合 coef_ :", np.round(m_weighted.coef_, 8))
print("按权重重复   拟合 coef_ :", np.round(m_repeat.coef_, 8))
print("coef_ 一致(allclose)     :", bool(np.allclose(m_weighted.coef_, m_repeat.coef_, atol=1e-8)))
print("intercept_ 一致          :", bool(np.allclose(m_weighted.intercept_, m_repeat.intercept_, atol=1e-8)))
print()
print("sqrt(sw) 前 5 个         :", np.round(np.sqrt(sw[:5]), 4))


sample_weight 拟合 coef_ : [ 0.99186924 -2.99988342  2.01393371]
按权重重复   拟合 coef_ : [ 0.99186924 -2.99988342  2.01393371]
coef_ 一致(allclose)     : True
intercept_ 一致          : True

sqrt(sw) 前 5 个         : [1.4142 1.     1.     1.7321 1.    ]


## 6. SGD 的配置台：字符串 → 枚举 → C 内循环（R8/R9/R14）

`SGDRegressor` 的 Python 部分几乎不做数值计算。真正的梯度更新在编译模块 `_plain_sgd32/64`（来自 `_sgd_fast`，本地包内只有 `.so`）。Python 侧只做三件事：

1. **`loss` 字符串 → Cython 的 `LossFunction` 对象**（`_get_loss_function`）；
2. **`penalty` / `learning_rate` 字符串 → 整数枚举**：
   `PENALTY_TYPES = {"none":0,"l1":1,"l2":2,"elasticnet":3}`、
   `LEARNING_RATE_TYPES = {"constant":1,"optimal":2,"invscaling":3,"adaptive":4,"pa1":5,"pa2":6}`；
3. **一次性把二十多个参数传进 C 层**（`fit_binary` / `_fit_regressor` 里的 `_plain_sgd(...)` 调用）。

一个有意思的耦合：`learning_rate='optimal'` 时要求 `alpha > 0`（`_more_validate_params` L145–159），因为最优步长 `η = 1/(α(t+t₀))` **直接拿 alpha 算** —— 正则强度兼职学习率尺度。

另一个容易忽略的差异：**`BaseSGDRegressor` 的默认与基类 `BaseSGD` 不同** —— 回归器默认 `invscaling / eta0=0.01 / power_t=0.25`（L1397–1399），基类默认 `optimal / eta0=0.0 / power_t=0.5`（L111–113）。

最后是「Python 配置 → C 数据视图」的桥梁 `make_dataset`（`_base.make_dataset` L136–186）：按 `X.dtype` 分派 `ArrayDataset32 / CSRDataset32` 或 `...64`，C 内循环靠它随机访问单样本。


In [8]:
import inspect

import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model._stochastic_gradient import (
    LEARNING_RATE_TYPES,
    PENALTY_TYPES,
    BaseSGD,
)

print("LEARNING_RATE_TYPES :", LEARNING_RATE_TYPES)
print("PENALTY_TYPES       :", PENALTY_TYPES)
print("-> 字符串在进入 C 层之前，先被翻译成整数枚举")

# 回归器默认 vs 基类默认
sig_reg = inspect.signature(SGDRegressor.__init__).parameters
sig_base = inspect.signature(BaseSGD.__init__).parameters
print()
for name in ("learning_rate", "eta0", "power_t"):
    print("  %-14s BaseSGD=%-12r SGDRegressor=%r" % (
        name, sig_base[name].default, sig_reg[name].default))
print("-> 回归器默认 invscaling / 0.01 / 0.25，与基类 optimal / 0.0 / 0.5 不同")

# learning_rate='optimal' 反向约束 alpha>0
X = np.random.RandomState(0).randn(50, 3)
y = X @ np.array([1.0, 2.0, 3.0])
try:
    SGDRegressor(learning_rate="optimal", alpha=0, max_iter=5).fit(X, y)
except ValueError as e:
    print()
    print("SGDRegressor(learning_rate='optimal', alpha=0).fit  ->  ValueError:")
    print("   ", str(e).strip())


LEARNING_RATE_TYPES : {'constant': 1, 'optimal': 2, 'invscaling': 3, 'adaptive': 4, 'pa1': 5, 'pa2': 6}
PENALTY_TYPES       : {'none': 0, 'l2': 2, 'l1': 1, 'elasticnet': 3}
-> 字符串在进入 C 层之前，先被翻译成整数枚举

  learning_rate  BaseSGD='optimal'    SGDRegressor='invscaling'
  eta0           BaseSGD=0.0          SGDRegressor=0.01
  power_t        BaseSGD=0.5          SGDRegressor=0.25
-> 回归器默认 invscaling / 0.01 / 0.25，与基类 optimal / 0.0 / 0.5 不同

SGDRegressor(learning_rate='optimal', alpha=0).fit  ->  ValueError:
    alpha must be > 0 since learning_rate is 'optimal'. alpha is used to compute the optimal learning rate.


In [9]:
import inspect

import numpy as np
from sklearn.linear_model._base import make_dataset
from sklearn.linear_model._stochastic_gradient import _get_plain_sgd_function

print("# _get_plain_sgd_function() —— 分派到 _sgd_fast 的 _plain_sgd32 / _plain_sgd64")
print(inspect.getsource(_get_plain_sgd_function))
print()
print("实际对象: float32 ->", _get_plain_sgd_function(np.float32).__name__,
      "| float64 ->", _get_plain_sgd_function(np.float64).__name__)

print()
print("# make_dataset() —— 按 X.dtype 分派 32/64 位 Dataset，并给出 intercept_decay")
print(inspect.getsource(make_dataset))


# _get_plain_sgd_function() —— 分派到 _sgd_fast 的 _plain_sgd32 / _plain_sgd64
def _get_plain_sgd_function(input_dtype):
    return _plain_sgd32 if input_dtype == np.float32 else _plain_sgd64


实际对象: float32 -> _plain_sgd32 | float64 -> _plain_sgd64

# make_dataset() —— 按 X.dtype 分派 32/64 位 Dataset，并给出 intercept_decay
def make_dataset(X, y, sample_weight, random_state=None):
    """Create ``Dataset`` abstraction for sparse and dense inputs.

    This also returns the ``intercept_decay`` which is different
    for sparse datasets.

    Parameters
    ----------
    X : array-like, shape (n_samples, n_features)
        Training data

    y : array-like, shape (n_samples, )
        Target values.

    sample_weight : numpy array of shape (n_samples,)
        The weight of each sample

    random_state : int, RandomState instance or None (default)
        Determines random number generation for dataset random sampling. It is not
        used for dataset shuffling.
        Pass an int for repro

## 7. SGD 的优化状态：t_ / ASGD / 验证掩码 / 截距衰减（R10/R11/R12/R13）

SGD 的进度不是「轮数」，而是**累计样本数**：`self.t_ += self.n_iter_ * X.shape[0]`（L1716），供 invscaling / optimal 的时间表使用。

关键细节：**`fit` 每次调用都会把 `t_` 重置为 1.0** —— 这行位于 `warm_start` 分支**之后**（L1540–1541），所以 **即使 `warm_start=True` 也照样重置**；只有 `partial_fit` 走 `_partial_fit`（L1506）、**不重置**，`_fit_regressor` 仅在 `t_` 不存在时才补 1.0（L1653–1654）。因此「跨调用累积」**只对 `partial_fit` 成立**。

`average=True` 是**平均 SGD（ASGD）**：除 `coef_` 外另存 `_standard_coef` / `_average_coef`，对迭代过程中的权重做**历史平均**；且只有满足 `average <= t_ − 1` 时才真正使用平均值（L1722–1728）。

验证集不是一份新数据，而是**同一数组上的 bool 掩码**（`_make_validation_split` L275 / L309），训练循环据此跳过；但验证回调**确实做了布尔索引** `X[validation_mask] / y[validation_mask] / sample_weight[validation_mask]`（`_make_validation_score_cb` L318–322）来构造评估对象。所以准确说法是「**训练侧不复制、验证侧按掩码提取子集**」，而不是「X 不复制」。

最后一个纯工程 corner case：稀疏数据的**截距更新要乘 `SPARSE_INTERCEPT_DECAY = 0.01`**（源码注释：*avoid intercept oscillation*），稠密则为 `1.0`（`_base.make_dataset` L178–184）。


In [10]:
import inspect

from sklearn.linear_model._stochastic_gradient import BaseSGD, BaseSGDRegressor

# _fit 开头：注意 t_ 是无条件重置的
lines = inspect.getsource(BaseSGDRegressor._fit).splitlines()
s = next(k for k, l in enumerate(lines) if "if self.warm_start" in l)
e = next(k for k, l in enumerate(lines) if "self._partial_fit(" in l)
print("# BaseSGDRegressor._fit() 开头 —— t_ 无条件重置为 1.0（位于 warm_start 分支之后）")
print("\n".join(lines[s:e + 1]))

# _fit_regressor：t_ 累加
lines = inspect.getsource(BaseSGDRegressor._fit_regressor).splitlines()
s = next(k for k, l in enumerate(lines) if "self.t_ += self.n_iter_ * X.shape[0]" in l)
print()
print("# BaseSGDRegressor._fit_regressor() —— t_ 累加的是「累计样本数」")
print("\n".join(lines[s:s + 3]))

print()
print("# BaseSGD._make_validation_score_cb() —— 验证回调按 bool mask 做布尔索引")
print(inspect.getsource(BaseSGD._make_validation_score_cb))


# BaseSGDRegressor._fit() 开头 —— t_ 无条件重置为 1.0（位于 warm_start 分支之后）
        if self.warm_start and getattr(self, "coef_", None) is not None:
            if coef_init is None:
                coef_init = self.coef_
            if intercept_init is None:
                intercept_init = self.intercept_
        else:
            self.coef_ = None
            self.intercept_ = None

        # Clear iteration count for multiple call to fit.
        self.t_ = 1.0

        self._partial_fit(

# BaseSGDRegressor._fit_regressor() —— t_ 累加的是「累计样本数」
        self.t_ += self.n_iter_ * X.shape[0]

        if self.average > 0:

# BaseSGD._make_validation_score_cb() —— 验证回调按 bool mask 做布尔索引
    def _make_validation_score_cb(
        self, validation_mask, X, y, sample_weight, classes=None
    ):
        if not self.early_stopping:
            return None

        return _ValidationScoreCallback(
            self,
            X[validation_mask],
            y[validation_mask],
            sample_weight[v

In [11]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model._base import SPARSE_INTERCEPT_DECAY, make_dataset

rng = np.random.RandomState(0)
X = rng.randn(394, 4)
y = X @ np.array([1.0, 2.0, 3.0, 4.0]) + 0.5

# --- t_：partial_fit 累积 vs fit 重置 ---
sgd = SGDRegressor(max_iter=1, tol=None, random_state=0)
sgd.partial_fit(X, y)
print("after partial_fit #1 : t_ =", sgd.t_)
sgd.partial_fit(X, y)
print("after partial_fit #2 : t_ =", sgd.t_, "  <- 延续累加")
sgd.fit(X, y)
print("after fit            : t_ =", sgd.t_, "  <- fit 无条件重置为 1.0 后再累加一轮样本数")

sgd_ws = SGDRegressor(max_iter=1, tol=None, random_state=0, warm_start=True)
sgd_ws.partial_fit(X, y)
print("warm_start 下 partial_fit -> t_ =", sgd_ws.t_)
sgd_ws.fit(X, y)
print("warm_start 下 fit         -> t_ =", sgd_ws.t_, "  <- warm_start=True 也照样重置")

# --- 验证掩码：同一数组上的 bool mask ---
mask_off = sgd._make_validation_split(y, sample_mask=np.ones_like(y, dtype=bool))
print()
print("early_stopping=False 时 validation_mask:", np.unique(mask_off), "（空掩码 -> 全用训练集）")
sgd_es = SGDRegressor(max_iter=1, tol=None, random_state=0,
                      early_stopping=True, validation_fraction=0.2)
mask_on = sgd_es._make_validation_split(y, sample_mask=np.ones_like(y, dtype=bool))
print("early_stopping=True  时 validation_mask: shape=%s dtype=%s 验证样本数=%d"
      % (mask_on.shape, mask_on.dtype, int(mask_on.sum())))

# --- average=True (ASGD) 与默认解的差异 ---
Xa = rng.randn(200, 5)
ya = Xa @ np.array([1.5, -2.0, 0.5, 3.0, -1.0]) + 0.7
avg = SGDRegressor(max_iter=30, tol=None, random_state=0, learning_rate="constant",
                   eta0=0.01, average=True).fit(Xa, ya)
plain = SGDRegressor(max_iter=30, tol=None, random_state=0, learning_rate="constant",
                     eta0=0.01, average=False).fit(Xa, ya)
print()
print("average=False coef_:", np.round(plain.coef_, 4))
print("average=True  coef_:", np.round(avg.coef_, 4))
print("两者不同:", bool(not np.allclose(avg.coef_, plain.coef_, atol=1e-9)))

# --- intercept_decay: dense 1.0 vs sparse 0.01 ---
sw = np.ones(X.shape[0])
ds_dense, decay_dense = make_dataset(X, y, sw, random_state=0)
ds_sparse, decay_sparse = make_dataset(csr_matrix(X), y, sw, random_state=0)
print()
print("SPARSE_INTERCEPT_DECAY =", SPARSE_INTERCEPT_DECAY)
print("稠密   make_dataset -> dataset:", type(ds_dense).__name__, " intercept_decay:", decay_dense)
print("稀疏   make_dataset -> dataset:", type(ds_sparse).__name__, " intercept_decay:", decay_sparse)
print("-> 稀疏时截距更新乘 0.01，避免截距震荡")


after partial_fit #1 : t_ = 395.0
after partial_fit #2 : t_ = 789.0   <- 延续累加
after fit            : t_ = 395.0   <- fit 无条件重置为 1.0 后再累加一轮样本数
warm_start 下 partial_fit -> t_ = 395.0
warm_start 下 fit         -> t_ = 395.0   <- warm_start=True 也照样重置

early_stopping=False 时 validation_mask: [False] （空掩码 -> 全用训练集）
early_stopping=True  时 validation_mask: shape=(394,) dtype=bool 验证样本数=79

average=False coef_: [ 1.4998 -1.9998  0.4999  2.9996 -0.9999]
average=True  coef_: [ 1.4736 -1.9732  0.491   2.936  -0.9753]
两者不同: True

SPARSE_INTERCEPT_DECAY = 0.01
稠密   make_dataset -> dataset: ArrayDataset64  intercept_decay: 1.0
稀疏   make_dataset -> dataset: CSRDataset64  intercept_decay: 0.01
-> 稀疏时截距更新乘 0.01，避免截距震荡


## 8. 小结

把两条主线串成一句：**「Python 分派台 + C 内循环」**。

- **OLS 的 Python 侧只做分派**：`positive` → NNLS、稀疏 → LSQR、稠密 → SVD（`lstsq`）；多目标不是新后端而是策略（原生 2D 或逐列 `Parallel`）；`rank_` / `singular_` 只是 `lstsq` 的「附带产物」，因此只在「稠密 + `positive=False`」路径上存在。分派依据是 **`dtype` / 形状 / 稀疏 / `positive` / 多目标**。
- **截距、权重、中心化都不改变算法**，而是通过**等价变换**把问题折成标准形式：去均值重构截距、`√w` 重缩放吸收权重、`LinearOperator` 隐式中心化保住稀疏性。
- **SGD 的 Python 侧是配置台**（字符串 → 枚举 → C 参数），真正的迭代在 `_sgd_fast`；优化状态（`t_`、ASGD、验证掩码、`intercept_decay`）由 Python 与 C 共同维护。
- **数值稳定性不由 sklearn 自己负责**，而是交给 scipy 后端（SVD / LSQR / NNLS）兜底 —— 这是「用成熟数值库，而不是手写正规方程」的直接后果。
